# Packet Signature Baseline

Builds a simple packet-sequence signature baseline for cross-day IoT device classification.


In [14]:
from collections import Counter, defaultdict
from pathlib import Path
import re
import csv
import subprocess




import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)


SEQUENCE_LEN = 12
LENGTH_BUCKET = 10
ACTIVE_PERCENTILE = 0.75
MIN_PREFIX_LEN = 2


In [16]:
IP_HEADER_RE = re.compile(
    r"^(?P<time>\d+\.\d+) IP .* proto (?P<proto>[A-Z]+) \((?P<proto_num>\d+)\), length (?P<length>\d+)\)"
)
ENDPOINT_RE = re.compile(r"^\s+(?P<src>\S+) > (?P<dst>\S+):")

def endpoint_to_ip(endpoint):
    endpoint = endpoint.rstrip(":,")
    parts = endpoint.split(".")
    if len(parts) >= 4 and all(part.isdigit() for part in parts[:4]):
        return ".".join(parts[:4])
    return endpoint

def load_device_mapping(mapping_path="device_mapping.csv"):
    ip_to_device = {}
    with open(mapping_path, newline="") as f:
        for row in csv.reader(f):
            if len(row) >= 2:
                ip_to_device[row[1].strip()] = row[0].strip()
    return ip_to_device

def iter_ip_packets_from_tcpdump(pcap_path):
    cmd = ["tcpdump", "-tt", "-n", "-v", "-r", str(pcap_path), "ip"]
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.DEVNULL,
        text=True,
    )

    current = None
    for line in proc.stdout:
        header = IP_HEADER_RE.match(line)
        if header:
            current = {
                "time": float(header.group("time")),
                "protocol": int(header.group("proto_num")),
                "length": int(header.group("length")),
            }
            continue

        endpoints = ENDPOINT_RE.match(line)
        if endpoints and current is not None:
            current["src_ip"] = endpoint_to_ip(endpoints.group("src"))
            current["dst_ip"] = endpoint_to_ip(endpoints.group("dst"))
            yield current
            current = None

    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError("tcpdump failed while reading {}".format(pcap_path))


## Packet Sequence Extraction

Packets are grouped into fixed time windows per mapped device. Each window keeps a short signature of packet direction and bucketed packet length. Cached CSV files are reused when available.


In [9]:
def bucket_length(length):
    return int(round(length / LENGTH_BUCKET) * LENGTH_BUCKET)


def format_sequence(tokens):
    return " ".join("{}:{}".format(direction, length) for direction, length in tokens)


def parse_sequence(sequence):
    tokens = []
    for token in sequence.split():
        direction, length = token.split(":")
        tokens.append((direction, int(length)))
    return tokens


def extract_sequences(day_dir, ip_to_device):

    grouped = defaultdict(lambda: {"tokens": [], "pkt_count": 0, "total_bytes": 0})
    first_time = None
    pcap_paths = sorted(Path(day_dir).glob("eth1-*"))

    for file_idx, pcap_path in enumerate(pcap_paths, start=1):
        if file_idx == 1 or file_idx % 48 == 0 or file_idx == len(pcap_paths):
            print("Extracting packet signatures from {} ({}/{})".format(day_dir, file_idx, len(pcap_paths)))

        for pkt in iter_ip_packets_from_tcpdump(pcap_path):
            src = pkt["src_ip"]
            dst = pkt["dst_ip"]
            if src in ip_to_device:
                device = ip_to_device[src]
                direction = "O"
            elif dst in ip_to_device:
                device = ip_to_device[dst]
                direction = "I"
            else:
                continue

            if first_time is None:
                first_time = pkt["time"]

            window_start = ((pkt["time"] - first_time) // 5) * 5
            entry = grouped[(window_start, device)]
            entry["pkt_count"] += 1
            entry["total_bytes"] += pkt["length"]
            if len(entry["tokens"]) < SEQUENCE_LEN:
                entry["tokens"].append((direction, bucket_length(pkt["length"])))

    rows = []
    for (window_start, device), values in grouped.items():
        if not values["tokens"]:
            continue
        rows.append(
            {
                "window_start": window_start,
                "device": device,
                "label": device,
                "pkt_count": values["pkt_count"],
                "total_bytes": values["total_bytes"],
                "sequence": format_sequence(values["tokens"]),
            }
        )

    if not rows:
        raise ValueError("No mapped device packets found in {}".format(day_dir))

    df = pd.DataFrame(rows).sort_values(["window_start", "device"])
    return df


## Filtering

The baseline keeps labels that have enough samples on both train and test days, then focuses on active windows using a packet-count threshold learned from the training day.


In [10]:
def filter_common_labels(train_df, test_df):
    train_counts = train_df["label"].value_counts()
    test_counts = test_df["label"].value_counts()
    labels = sorted(
        set(train_counts[train_counts >= 10].index)
        & set(test_counts[test_counts >= 10].index)
    )
    return (
        train_df[train_df["label"].isin(labels)].copy(),
        test_df[test_df["label"].isin(labels)].copy(),
        labels,
    )


def active_threshold(train_df):
    return max(1, int(train_df["pkt_count"].quantile(ACTIVE_PERCENTILE)))


def keep_active_windows(train_df, test_df):
    threshold = active_threshold(train_df)
    return (
        train_df[train_df["pkt_count"] >= threshold].copy(),
        test_df[test_df["pkt_count"] >= threshold].copy(),
        threshold,
    )


## Signature Model

Training records exact packet signatures and all prefixes. Prediction first tries an exact match, then the longest matching prefix, then falls back to the most common training label.


In [11]:
def majority_label(counter):
    return counter.most_common(1)[0][0]


def build_signatures(train_df):
    exact = defaultdict(Counter)
    prefixes = defaultdict(Counter)
    prior = Counter(train_df["label"])

    for row in train_df.itertuples(index=False):
        tokens = tuple(parse_sequence(row.sequence))
        exact[tokens][row.label] += 1
        for prefix_len in range(MIN_PREFIX_LEN, len(tokens) + 1):
            prefixes[tokens[:prefix_len]][row.label] += 1

    return {"exact": exact, "prefixes": prefixes, "prior": prior}


def predict_one(sequence, signatures):
    tokens = tuple(parse_sequence(sequence))

    if tokens in signatures["exact"]:
        return majority_label(signatures["exact"][tokens])

    for prefix_len in range(len(tokens), MIN_PREFIX_LEN - 1, -1):
        prefix = tokens[:prefix_len]
        if prefix in signatures["prefixes"]:
            return majority_label(signatures["prefixes"][prefix])

    return majority_label(signatures["prior"])


## Evaluation

The evaluator returns summary metrics, a classification report, and a confusion matrix for one train/test direction.


In [12]:
def evaluate_signature_baseline(name, train_df, test_df):
    train_df, test_df, labels = filter_common_labels(train_df, test_df)
    train_df, test_df, threshold = keep_active_windows(train_df, test_df)
    train_df, test_df, labels = filter_common_labels(train_df, test_df)

    signatures = build_signatures(train_df)
    y_true = test_df["label"].tolist()
    y_pred = [predict_one(sequence, signatures) for sequence in test_df["sequence"]]

    precision, recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    report = classification_report(y_true, y_pred, zero_division=0)
    matrix = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=labels),
        index=labels,
        columns=labels,
    )

    return {
        "experiment": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": macro_f1,
        "n_train": len(train_df),
        "n_test": len(test_df),
        "n_devices": len(labels),
        "active_pkt_threshold": threshold,
        "report": report,
        "confusion_matrix": matrix,
    }


## Run Cross-Day Baseline

This final cell mirrors the original script `main()` function. It loads or extracts cached sequence tables, evaluates both cross-day directions, saves result tables, and prints the comparison.


In [ ]:
ip_to_device = load_device_mapping()

day20 = extract_sequences(
    "2018/03/20",
    ip_to_device
)
day21 = extract_sequences(
    "2018/03/21",
    ip_to_device
)

results = [
    evaluate_signature_baseline(
        "Packet signature 2018-03-20 -> 2018-03-21",
        day20,
        day21,
    ),
    evaluate_signature_baseline(
        "Packet signature 2018-03-21 -> 2018-03-20",
        day21,
        day20,
    ),
]

comparison = pd.DataFrame(
    [
        {
            "experiment": result["experiment"],
            "accuracy": result["accuracy"],
            "macro_precision": result["macro_precision"],
            "macro_recall": result["macro_recall"],
            "macro_f1": result["macro_f1"],
            "n_train": result["n_train"],
            "n_test": result["n_test"],
            "n_devices": result["n_devices"],
            "active_pkt_threshold": result["active_pkt_threshold"],
        }
        for result in results
    ]
)
# comparison.to_csv(output_dir / "packet_signature_baseline_results.csv", index=False)

# print("\nPacket-signature baseline comparison")
# print(comparison.to_string(index=False))
comparison



Extracting packet signatures from 2018/03/20 (1/288)
Extracting packet signatures from 2018/03/20 (48/288)
Extracting packet signatures from 2018/03/20 (96/288)
Extracting packet signatures from 2018/03/20 (144/288)
Extracting packet signatures from 2018/03/20 (192/288)
Extracting packet signatures from 2018/03/20 (240/288)
Extracting packet signatures from 2018/03/20 (288/288)
Extracting packet signatures from 2018/03/21 (1/288)
Extracting packet signatures from 2018/03/21 (48/288)
Extracting packet signatures from 2018/03/21 (96/288)
Extracting packet signatures from 2018/03/21 (144/288)
Extracting packet signatures from 2018/03/21 (192/288)
Extracting packet signatures from 2018/03/21 (240/288)
Extracting packet signatures from 2018/03/21 (288/288)


,experiment,accuracy,macro_precision,macro_recall,macro_f1,n_train,n_test,n_devices,active_pkt_threshold
0,Packet signature train 2018-03-20 -> test 2018...,0.895400,0.598799,0.581689,0.545193,62694,73786,29,47
1,Packet signature train 2018-03-21 -> test 2018...,0.865059,0.564058,0.373979,0.356428,72299,60708,28,56
